In [ ]:
# ── Notebook parameters ─────────────────────────────────────────
# Defaults for interactive use; papermill / extract_notebook_figures.py
# override these via a cell injected right below this one.

# ── Experiment toggles ──────────────────────────────────────────
# RANK_* is the number R of mixing tokens. R is chosen per call, not at
# construction: the same model is pushed forward with rank=RANK_*.
# BASE_DICTIONARY_* is "Fourier" or "Haar" (both infinite bases).
BASE_DICTIONARY_1D           = "Haar"
MIXING_FIELD_ARCHITECTURE_1D = "nerf"
NUM_LAYERS_1D                = 8
RANK_1D                      = 8

BASE_DICTIONARY_2D           = "Fourier"
MIXING_FIELD_ARCHITECTURE_2D = "nerf"
NUM_LAYERS_2D                = 8
RANK_2D                      = 10

BASE_DICTIONARY_3CH           = "Haar"
MIXING_FIELD_ARCHITECTURE_3CH = "nerf"
NUM_LAYERS_3CH                = 8
RANK_3CH                      = 10

# Changing Bases with a Causal Sequence Mixer

This notebook demonstrates the sequence-mixer change of basis (`EulerianIsometry`). The input is always an **ordered prefix** $e_1,\dots,e_K$ of a reference dictionary (its first $K$ atoms by descending PMF, `get_prefix(coords, K)`), with $K \ge R$. The number of mixing tokens $R$ is not part of the model: it is chosen per call (`rank=R` on `pushforward`), so one trained model serves every $R \le K$. The isometry is a stack of $L$ transformer-style layers, and each layer is **one exact rotation** $Q_\ell$ applied to every function of the prefix:

1. **Mixing functions.** $R$ mixing tokens $q_r$ condition a shared neural field, giving $u_r(x) = \mathrm{MLP}(x, q_r)$.
2. **Causal cross-attention.** The mixing tokens attend to the function tokens $p_k$ with a causal mask ($k \le r$), giving a lower-triangular $A \in \mathbb{R}^{R\times R}$ and $a_r = \sum_{k\le r} A_{rk}\, e_k$.
3. **Skew generator and Cayley step.** With the analysis maps $E f = (\langle e_k, f\rangle)_k$ and $U f = (\langle u_r, f\rangle)_r$,
$$L = U^\ast A E - E^\ast A^\top U, \qquad L^\ast = -L, \qquad Q_\ell = \bigl(I - \tfrac12 L\bigr)^{-1}\bigl(I + \tfrac12 L\bigr).$$

$Q_\ell$ is orthogonal, so the Gram matrix $G_{jk} = \langle e_j, e_k\rangle$ of the prefix is preserved exactly. The prefix need not be orthonormal: the method only relies on $G$ being preserved (Fourier and Haar happen to be orthonormal). Because the mask makes every layer depend only on $e_1,\dots,e_R$, appending more atoms to the prefix never changes the earlier outputs (prefix faithfulness): a longer prefix is how atoms beyond the first $K$ are transported. Arbitrary data $f$ is analysed by inner products $\langle f, T e_k\rangle$ with the pushed prefix; expressing $f$ in the learned basis solves against the preserved Gram matrix (`project_onto_span`). The preserved inner product is the weighted real product

$$\langle f, g\rangle = \frac{1}{N}\sum_n e^{\ell_n}\, f(x_n)\cdot g(x_n).$$

In [ ]:
%load_ext autoreload
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import functools
sys.path.append('..')  # for notebook_helpers in notebooks/

from notebook_helpers import (
    visualize_mixing_layers,
    plot_signal_1d,
    show_scalar_2d,
)
from infidictionary.neural_isometries import (
    EulerianIsometry, ChannelOrthogonalIsometry, ChainedIsometry,
)
from infidictionary.networks import NerfConditionalField
from infidictionary.dictionaries import FourierDictionary, HaarWaveletDictionary
from infidictionary.domain_samplers import SquareSampler
from infidictionary.utils import pairwise_inner_product, norm2, project_onto_span

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64)


In [ ]:
def make_dictionary(kind: str, domain_dim: int, num_channels: int = 1):
    """Construct a real Fourier or Haar starting dictionary."""
    if kind == "Fourier":
        return FourierDictionary(domain_dim=domain_dim, num_channels=num_channels, steepness=2.0)
    if kind == "Haar":
        return HaarWaveletDictionary(domain_dim=domain_dim, num_channels=num_channels)
    raise ValueError(f"unknown dictionary kind: {kind}; use Fourier or Haar")


def make_field_partial(architecture: str):
    if architecture == "nerf":
        return functools.partial(
            NerfConditionalField,
            hidden_dims=[256, 256, 256],
            activation=torch.nn.SiLU,
            nerf_n_levels=8,
            nerf_n_base=64,
            nerf_freq_min=1.0,
            nerf_freq_max=24.0,
        )
    raise ValueError(f"unknown field architecture: {architecture}")


def make_isometry(
    coords_dim: int,
    architecture: str,
    num_layers: int,
    channels_dim: int = 1,
    gradient_checkpointing: bool = False,
):
    eulerian_partial = functools.partial(
        EulerianIsometry,
        num_layers=num_layers,
        scalar_field_partial=make_field_partial(architecture),
        d_model=64,
        ffn_hidden=128,
        gradient_checkpointing=gradient_checkpointing,
    )
    eulerian = eulerian_partial(coords_dim=coords_dim, channels_dim=channels_dim).to(device)
    # A scalar field has no channels to mix. Multi-channel examples always
    # begin with a Haar-random channel rotation before the sequence mixer.
    if channels_dim == 1:
        return eulerian, eulerian

    chain = ChainedIsometry(
        coords_dim=coords_dim,
        channels_dim=channels_dim,
        isometries_partial=[
            functools.partial(ChannelOrthogonalIsometry, hidden_dims=(64, 64)),
            eulerian_partial,
        ],
    ).to(device)
    return chain, chain.isometries[1]


def layer_selection(num_layers: int):
    """First, middle, and last layer (0-based, deduplicated)."""
    return sorted({0, num_layers // 2, num_layers - 1})


# Prefix lengths. Atoms are addressed by their prefix position: "atom k" is
# row k of get_prefix(coords, K). N_DEMO leading atoms are plotted; the Gram
# checks use the first GRAM_PREFIX[domain_dim] atoms. Every pushed prefix
# holds at least R atoms, as pushforward requires.
N_DEMO = 5
GRAM_PREFIX = {1: 16, 2: 48}


def prefix_length(count: int, rank: int) -> int:
    """Length of the prefix to push so that it covers ``count`` atoms and K >= R."""
    return max(count, rank)

## A Simple 1-D Experiment

The smallest end-to-end example on $[0,1]$. The initial basis (`BASE_DICTIONARY_1D`), the mixing-field architecture, and the number of layers come from the parameters cell. For multi-channel examples, the notebook automatically prepends a `ChannelOrthogonalIsometry`; a one-channel field has nothing to mix.

We push the ordered prefix $e_1,\dots,e_K$ (`get_prefix`, $K = \max(5, R)$) with `rank=R`; its first $R$ atoms build the rotations. The model has no fixed rank: $R$ is chosen per call (`RANK_1D`). We show the first five prefix atoms before, after half of the layers (`num_layers_to_apply`), and after all $L$ layers.

In [ ]:
%autoreload 2
torch.manual_seed(0)

basis_1d = make_dictionary(BASE_DICTIONARY_1D, domain_dim=1)
rank_1d = RANK_1D   # R, passed per call
isometry_1d, eulerian_1d = make_isometry(
    coords_dim=1, architecture=MIXING_FIELD_ARCHITECTURE_1D,
    num_layers=NUM_LAYERS_1D,
)
isometry_1d.eval()
K_demo_1d = prefix_length(N_DEMO, rank_1d)
pmfs_1d = basis_1d.get_prefix_pmfs(K_demo_1d)

N_1d = 256
x_1d = SquareSampler(domain_dim=1, stratified=True, add_noise=True).sample(N_1d).to(device)   # (N, 1) on [0,1]
lad_1d = torch.zeros(N_1d, device=device)
src_1d = basis_1d.get_prefix(x_1d, K_demo_1d)                                         # (K, N, 1) ordered prefix

half_1d = NUM_LAYERS_1D // 2
with torch.no_grad():
    _, _, mid_1d = isometry_1d.pushforward(x_1d, lad_1d, src_1d, rank=rank_1d, num_layers_to_apply=half_1d)
    _, _, tgt_1d = isometry_1d.pushforward(x_1d, lad_1d, src_1d, rank=rank_1d)

# Left = before, middle = after half the layers, right = after all layers.
for a in range(N_DEMO):
    fig, (axb, axm, axa) = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    plot_signal_1d(x_1d[:, 0], src_1d[a], title='before', ax=axb)
    plot_signal_1d(x_1d[:, 0], mid_1d[a], title=f'after {half_1d} of {NUM_LAYERS_1D} layers', ax=axm)
    plot_signal_1d(x_1d[:, 0], tgt_1d[a], title=f'after all {NUM_LAYERS_1D} layers', ax=axa)
    fig.suptitle(f'1-D prefix atom $e_{{{a + 1}}}$  (PMF={pmfs_1d[a].item():.2g}, {BASE_DICTIONARY_1D})')
    plt.show()

### 1-D Gram check

Each layer is orthogonal, so $G_{ab} = \langle Te_a, Te_b\rangle$ over the first $K$ prefix atoms must equal the initial $\langle e_a, e_b\rangle$. The method does not need the prefix to be orthonormal, only that $G$ is preserved; Fourier and Haar happen to be orthonormal, so here the initial Gram is close to the identity. We sample the quadrature points **independently** for the two Grams — matching heatmaps reflect the $L^2$ isometry, not a shared point set.

In [ ]:
%autoreload 2
# Independent quadrature points for the initial vs the pushed Gram.
x_init = SquareSampler(domain_dim=1, stratified=True, add_noise=True).sample(N_1d).to(device)
x_push = SquareSampler(domain_dim=1, stratified=True, add_noise=True).sample(N_1d).to(device)
K_gram_1d = prefix_length(GRAM_PREFIX[1], rank_1d)

atoms_init_1d = basis_1d.get_prefix(x_init, K_gram_1d)
with torch.no_grad():
    _, _, atoms_push_1d = isometry_1d.pushforward(x_push, lad_1d, basis_1d.get_prefix(x_push, K_gram_1d), rank=rank_1d)
    G_init_1d   = pairwise_inner_product(atoms_init_1d, atoms_init_1d, lad_1d)
    G_pushed_1d = pairwise_inner_product(atoms_push_1d, atoms_push_1d, lad_1d)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
vmin = min(G_init_1d.min().item(), G_pushed_1d.min().item())
vmax = max(G_init_1d.max().item(), G_pushed_1d.max().item())
for ax, G, title in zip(axes, [G_init_1d, G_pushed_1d, G_pushed_1d - G_init_1d],
                        ['G initial', 'G pushed (resampled)', '|G pushed - G init|']):
    im = ax.imshow(G.detach().cpu(), cmap='viridis', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title); ax.set_xlabel('prefix position b'); ax.set_ylabel('prefix position a')
fig.suptitle(f'1-D Gram matrices of the first {K_gram_1d} prefix atoms — pushed basis (on resampled points) matches the initial Gram')
plt.show()
print('max |G_pushed - G_init| =', (G_pushed_1d - G_init_1d).abs().max().item())

### 1-D mixing layers

The mixing functions and attention live on the **Eulerian link** (`eulerian_1d`); the channel-orthogonal link has none. For the first, middle, and last layer, the top rows show the mixing functions $u_r^{(\ell)}$ (real 1-D signals) and the bottom row shows the causal attention $A^{(\ell)}$, which is lower triangular.

In [ ]:
%autoreload 2
# u_r and A for selected layers of the Eulerian link. The chain itself is not
# an EulerianIsometry, so we pass its Eulerian link (eulerian_1d) directly.
coords_vis_1d = SquareSampler(domain_dim=1, stratified=True, add_noise=True).sample(200).to(device)
visualize_mixing_layers(eulerian_1d, coords_vis_1d, rank_1d, layers=layer_selection(NUM_LAYERS_1D))

## 2-D Experiment

The same pipeline on $[0,1]^2$, single channel. The basis, rank $R$, mixing-field architecture, and number of layers are all set by the `*_2D` parameters.

Build the dictionary and the isometry from the toggles. The isometry has no fixed rank; $R$ is chosen per call (`rank=`) and is capped at the dictionary size, since the prefix needs $K \ge R$:

In [ ]:
%autoreload 2
torch.manual_seed(0)
basis_2d = make_dictionary(BASE_DICTIONARY_2D, domain_dim=2)
rank_2d = RANK_2D   # R, passed per call
isometry, eulerian_2d = make_isometry(
    coords_dim=2, architecture=MIXING_FIELD_ARCHITECTURE_2D,
    num_layers=NUM_LAYERS_2D, gradient_checkpointing=True,
)
isometry.eval()

Push the ordered prefix through all layers (its first $R$ atoms build the rotations) and show the leading atoms before/after:

In [ ]:
%autoreload 2
torch.manual_seed(0)
K_demo_2d = prefix_length(N_DEMO, rank_2d)
pmfs_2d = basis_2d.get_prefix_pmfs(K_demo_2d)

N_per_dim = 64
xy = SquareSampler(stratified=True, add_noise=True).sample(N_per_dim).to(device)
lad_xy = torch.zeros(xy.shape[0], device=device)
src_field = basis_2d.get_prefix(xy, K_demo_2d)   # (K, N, 1) ordered prefix

with torch.no_grad():
    _, _, tgt_field = isometry.pushforward(xy, lad_xy, src_field, rank=rank_2d)

# Single-channel fields are shown as scalar heatmaps.
n_show = N_DEMO
fig, axes = plt.subplots(n_show, 2, figsize=(7, 3.0 * n_show), constrained_layout=True, squeeze=False)
for i in range(n_show):
    show_scalar_2d(axes[i, 0], xy, src_field[i], title=f'before  ($e_{{{i + 1}}}$, PMF={pmfs_2d[i].item():.2g})')
    show_scalar_2d(axes[i, 1], xy, tgt_field[i], title=f'after {NUM_LAYERS_2D} layers')
fig.suptitle(f'2-D prefix atoms before / after pushforward  ({BASE_DICTIONARY_2D})')
plt.show()

Inner-product (Gram) check over the first $K$ prefix atoms, with quadrature points sampled **independently** for the original and pushed bases — the heatmaps should still match (the transform is an isometry), up to Monte-Carlo error between the two point clouds:

In [ ]:
%autoreload 2
# Independent quadrature points for the original vs the pushed basis.
coords_init = SquareSampler(stratified=True, add_noise=True).sample(N_per_dim).to(device)
coords_push = SquareSampler(stratified=True, add_noise=True).sample(N_per_dim).to(device)
lad_push = torch.zeros(coords_push.shape[0], device=device)
K_gram_2d = prefix_length(GRAM_PREFIX[2], rank_2d)
atoms_init = basis_2d.get_prefix(coords_init, K_gram_2d)
with torch.no_grad():
    _, _, atoms_push = isometry.pushforward(coords_push, lad_push, basis_2d.get_prefix(coords_push, K_gram_2d), rank=rank_2d)

# Gram matrices are real; show <.,.>.
G_init = pairwise_inner_product(atoms_init, atoms_init)
G_pushed = pairwise_inner_product(atoms_push, atoms_push)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
m = axes[1].imshow(G_init.detach().cpu(), cmap='viridis'); fig.colorbar(m, ax=axes[1], label='inner product')
axes[1].set_title('Initial basis'); axes[1].set_xlabel('prefix position b'); axes[1].set_ylabel('prefix position a')
m = axes[2].imshow(G_pushed.detach().cpu(), cmap='viridis'); fig.colorbar(m, ax=axes[2], label='inner product')
axes[2].set_title('Pushed basis (resampled)'); axes[2].set_xlabel('prefix position b'); axes[2].set_ylabel('prefix position a')
axes[0].hist((G_init - G_pushed).abs().flatten().detach().cpu(), bins=30)
axes[0].set_title('|G initial − G pushed|'); axes[0].set_xlabel('|difference|'); axes[0].set_ylabel('count')
plt.show()
print('max |G_pushed - G_init| =', (G_init - G_pushed).abs().max().item())

## Mixing functions $u_r$ and causal attention $A$

On the Eulerian link (`eulerian_2d`), for the first, middle, and last layer: the top rows are the mixing functions $u_r^{(\ell)}$, and the last row is the lower-triangular attention $A^{(\ell)}$ that forms $a_r = \sum_{k\le r} A_{rk} e_k$. Both depend on the parameters only (the tokens never read the prefix).

In [ ]:
%autoreload 2
# Mixing functions live on the Eulerian link; pass eulerian_2d (the chain itself
# is not an EulerianIsometry).
coords_vis = SquareSampler(stratified=True, add_noise=True).sample(80).to(device)
visualize_mixing_layers(eulerian_2d, coords_vis, rank_2d, layers=layer_selection(NUM_LAYERS_2D))

### Coefficients in the learned basis and prefix faithfulness

The isometry has no inverse call. To express data $f$ in the learned basis we push the prefix forward and take inner products, $c_k = \langle f, Te_k\rangle$ for $k \le K$. The prefix need not be orthonormal (and on a finite point set even an orthonormal dictionary is only approximately so), so the reconstruction solves against the Gram matrix $G_{jk} = \langle Te_j, Te_k\rangle$: $\hat f = \sum_{k\le K} (G^{+}c)_k\, Te_k$ is the orthogonal projection of $f$ onto the span of the pushed prefix (`project_onto_span`), and $\|\hat f\|^2 = \langle f, \hat f\rangle$ is the energy it captures. Three exact checks follow:

* **Isometry on the sample points.** The pushed Gram equals the Gram of $e_1,\dots,e_K$ on the same points.
* **Analysis/synthesis round trip.** A function synthesized in the pushed basis is its own projection onto the pushed prefix.
* **Prefix faithfulness.** Pushing a length-$R$ prefix and a length-$K$ prefix gives the same first $R$ atoms, so extending the prefix never changes the earlier learned atoms.

We visualize $f = e_1, e_2, e_3$, its projection $\hat f$ onto the pushed prefix, and the residual $f - \hat f$ (the part of $f$ outside that span, not a numerical error: the rotation moves energy out of the span of $e_1,\dots,e_K$).

In [ ]:
%autoreload 2
coords = SquareSampler(stratified=True, add_noise=True).sample(64).to(device)
lad_rt = torch.zeros(coords.shape[0], device=device)
K_rec = prefix_length(GRAM_PREFIX[2], rank_2d)
prefix_rt = basis_2d.get_prefix(coords, K_rec)   # (K, N, 1) ordered prefix
f = prefix_rt[:3]                                 # data to express: e_1, e_2, e_3

with torch.no_grad():
    _, _, pushed_rt = isometry.pushforward(coords, lad_rt, prefix_rt, rank=rank_2d)
    _, _, pushed_short = isometry.pushforward(coords, lad_rt, prefix_rt[:rank_2d], rank=rank_2d)

G_rt = pairwise_inner_product(pushed_rt, pushed_rt, lad_rt)    # (K, K) Gram of the pushed prefix
f_hat, f_hat_energy = project_onto_span(f, pushed_rt, lad_rt)  # projection via G^+ <T e, f>, and ||f_hat||^2
diff = f - f_hat

f_energy = norm2(f, lad_rt).clamp(min=1e-12)
rel = norm2(diff, lad_rt) / f_energy
captured = f_hat_energy / f_energy

# Exact checks: isometry on the push points, analysis/synthesis round trip, prefix faithfulness.
G_prefix = pairwise_inner_product(prefix_rt, prefix_rt, lad_rt)
c_true = torch.randn(4, K_rec, device=device, dtype=pushed_rt.dtype)
g = torch.einsum('ak,knc->anc', c_true, pushed_rt)   # synthesized in the pushed basis
g_hat, _ = project_onto_span(g, pushed_rt, lad_rt)
round_trip = (norm2(g - g_hat, lad_rt) / norm2(g, lad_rt)).max().sqrt().item()
faithfulness = (pushed_rt[:rank_2d] - pushed_short).abs().max().item()

titles = [
    'Data $f$',
    'Learned atom $T e_i$',
    'Projection $\\hat f$ onto pushed prefix',
    'Residual $f - \\hat f$',
]
fig, axes = plt.subplots(len(f), 4, figsize=(18, 4 * len(f)), constrained_layout=True, squeeze=False)
for i in range(len(f)):
    signal_scale = torch.stack([f[i].abs().max(), pushed_rt[i].abs().max(), f_hat[i].abs().max()]).max().item()
    signal_scale = max(signal_scale, 1e-12)
    row = [f[i], pushed_rt[i], f_hat[i], diff[i]]
    for j, (field, title) in enumerate(zip(row, titles)):
        hb = show_scalar_2d(
            axes[i, j],
            coords,
            field,
            title=title,
            vmin=-signal_scale,
            vmax=signal_scale,
        )
        cbar = fig.colorbar(hb, ax=axes[i, j], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=7)
    axes[i, 0].set_title(f'Data $f = e_{{{i + 1}}}$')
    axes[i, 1].set_title(f'Learned atom $T e_{{{i + 1}}}$')
    axes[i, 3].set_title(
        f'Residual on signal scale\ncaptured={captured[i].item():.3f}, rel. residual={rel[i].item():.2e}',
        fontsize=9,
    )
fig.suptitle(f'Coefficients and projection onto the first {K_rec} pushed prefix atoms (one shared color scale per row)')
plt.show()

print('Captured energy fraction per function (<= 1):', [f'{c:.3f}' for c in captured.tolist()])
print('Relative residual per function:', [f'{r:.2e}' for r in rel.tolist()])
print('max |G_pushed - G_prefix| on the push points =', (G_rt - G_prefix).abs().max().item())
print('synthesis/analysis relative round-trip error =', round_trip)
print(f'prefix faithfulness |T(e_1..e_{K_rec})[:R] - T(e_1..e_R)| =', faithfulness)

---
## Multichannel (C = 3) Eulerian Isometry — per-channel views

Atoms now live in $L^2([0,1]^2, \mathbb{R}^3)$. The chain first applies a spatially varying channel rotation, then the sequence mixer; the whole map mixes energy across channels while preserving the full $\mathbb{R}^3$-valued inner product. The whole prefix goes through the chain, so the mixer is built from the prefix as it arrives there (`ChainedIsometry` forwards `rank=` to the Eulerian link). We visualize **each output channel separately** with a real scalar colour scale rather than packing the three channels into a single RGB image.

In [ ]:
%autoreload 2
# The chained map applies a spatially varying real channel rotation first,
# then the sequence-mixer basis transport. Keep the Eulerian link for
# visualising its mixing layers below.
torch.manual_seed(42)
fourier_3ch = make_dictionary(BASE_DICTIONARY_3CH, domain_dim=2, num_channels=3)
rank_3ch = RANK_3CH   # R, passed per call
isometry_3ch, eulerian_3ch = make_isometry(
    coords_dim=2,
    architecture=MIXING_FIELD_ARCHITECTURE_3CH,
    num_layers=NUM_LAYERS_3CH,
    channels_dim=3,
)
isometry_3ch.eval()

N_3ch = 64
coords_3ch = SquareSampler(stratified=True, add_noise=True).sample(N_3ch).to(device)
lad_3ch = torch.zeros(coords_3ch.shape[0], device=device)

### Per-channel pushforward

The first non-constant prefix atoms $e_4, e_5, e_6$ are the same spatial atom seeded in channels 0, 1 and 2 (the three channel copies share a PMF, and the constants $e_1, e_2, e_3$ come first). After the cross-channel orthogonal map, energy can spread across all three channels — visible because each channel is rendered on its own panel.

In [ ]:
%autoreload 2
# Prefix positions 4..6 (1-based): one non-constant spatial atom per channel.
test_positions_3ch = [3, 4, 5]
K_3ch = prefix_length(max(test_positions_3ch) + 1, rank_3ch)
prefix_3ch = fourier_3ch.get_prefix(coords_3ch, K_3ch)   # (K, N, 3) ordered prefix
pmfs_3ch = fourier_3ch.get_prefix_pmfs(K_3ch)

with torch.no_grad():
    _, _, prefix_3ch_pushed = isometry_3ch.pushforward(coords_3ch, lad_3ch, prefix_3ch, rank=rank_3ch)
atoms_3ch = prefix_3ch[test_positions_3ch]                # (3, N, 3) real
atoms_3ch_pushed = prefix_3ch_pushed[test_positions_3ch]

C = 3
# One symmetric scale for every atom and channel: panel colour now encodes
# comparable signed magnitude instead of each hexbin auto-scaling.
channel_limit = torch.cat([atoms_3ch, atoms_3ch_pushed], dim=0).abs().max().item()
channel_limit = max(channel_limit, 1e-12)
for a, position in enumerate(test_positions_3ch):
    fig, axes = plt.subplots(2, C, figsize=(3.4 * C, 7.0), constrained_layout=True)
    image = None
    for c in range(C):
        image = show_scalar_2d(
            axes[0, c], coords_3ch, atoms_3ch[a][:, c:c + 1], title=f'before . channel {c}',
            cmap='coolwarm', vmin=-channel_limit, vmax=channel_limit,
        )
        show_scalar_2d(
            axes[1, c], coords_3ch, atoms_3ch_pushed[a][:, c:c + 1], title=f'after . channel {c}',
            cmap='coolwarm', vmin=-channel_limit, vmax=channel_limit,
        )
    seeded_channel = int(atoms_3ch[a].square().sum(dim=0).argmax())
    fig.suptitle(
        f'prefix atom $e_{{{position + 1}}}$ (PMF={pmfs_3ch[position].item():.2g}, seeded in channel {seeded_channel})'
        '  —  top: before, bottom: after'
    )
    fig.colorbar(image, ax=axes.ravel().tolist(), label='field value (shared scale)')
    plt.show()

### Gram (isometry) check

The $\mathbb{R}^3$-valued Gram $G_{ab} = \langle Te_a, Te_b\rangle$ over the first $K$ prefix atoms must match the initial Gram after the pushforward, even though individual atoms mix across channels. As in the 1-D case we sample the quadrature points **independently** for the initial and the pushed Gram, so matching heatmaps reflect the $L^2$ isometry rather than a shared point set.

In [ ]:
%autoreload 2
# Independent quadrature points for the initial vs the pushed Gram.
coords_init = SquareSampler(stratified=True, add_noise=True).sample(N_3ch).to(device)
coords_push = SquareSampler(stratified=True, add_noise=True).sample(N_3ch).to(device)
lad_init = torch.zeros(coords_init.shape[0], device=device)
lad_push = torch.zeros(coords_push.shape[0], device=device)
K_gram_3ch = prefix_length(GRAM_PREFIX[2], rank_3ch)

atoms_init = fourier_3ch.get_prefix(coords_init, K_gram_3ch)
with torch.no_grad():
    _, _, atoms_push = isometry_3ch.pushforward(
        coords_push, lad_push, fourier_3ch.get_prefix(coords_push, K_gram_3ch), rank=rank_3ch,
    )
    G_init   = pairwise_inner_product(atoms_init, atoms_init, lad_init)
    G_pushed = pairwise_inner_product(atoms_push, atoms_push, lad_push)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
vmin = min(G_init.min().item(), G_pushed.min().item())
vmax = max(G_init.max().item(), G_pushed.max().item())
for ax, G, ttl in zip(axes, [G_init, G_pushed, G_pushed - G_init],
                      ['G initial', 'G pushed (resampled)', '|G pushed - G init|']):
    im = ax.imshow(G.detach().cpu(), cmap='viridis', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); ax.set_title(ttl)
    ax.set_xlabel('prefix position b'); ax.set_ylabel('prefix position a')
fig.suptitle(f'3-channel Gram matrices of the first {K_gram_3ch} prefix atoms — pushed basis (on resampled points) matches the initial Gram')
plt.show()
print('max |G_pushed - G_init| =', (G_pushed - G_init).abs().max().item())

### Mixing functions on a specific channel

`visualize_mixing_layers` takes the per-call rank $R$ and a `channel` argument, so you can inspect the mixing functions $u_r^{(\ell)}$ on a chosen output channel of the multi-channel isometry.

In [ ]:
%autoreload 2
N_vis_3ch = 56
coords_vis_3ch = SquareSampler(stratified=True, add_noise=True).sample(N_vis_3ch).to(device)

# inspect the mixing functions restricted to channel 1
visualize_mixing_layers(eulerian_3ch, coords_vis_3ch, rank_3ch, layers=layer_selection(NUM_LAYERS_3CH), channel=1)